# ZapalloAI — Notebook 03: Entrenamiento YOLOv11n-cls

**Universidad de las Fuerzas Armadas ESPE**  
Estudiantes: César Loor, Camilo Orrico  
Docente: Ing. Doris Chicaiza

---
## IMPORTANTE: Antes de ejecutar
Si acabas de instalar PyTorch con CUDA, **reinicia el kernel primero**:
- VS Code: `Kernel` → `Restart Kernel`
- Jupyter: `Kernel` → `Restart`

Sin reiniciar, Python seguirá usando la versión anterior sin CUDA.

---
## Modos
- `COLAB_MODE = False` → **Local con GTX 1650** (dataset en `model/data/processed/`)
- `COLAB_MODE = True` → Google Colab T4

In [ ]:
# ── CELDA 1: Verificar GPU — EJECUTAR PRIMERO ─────────────────────
# Si dice 'CUDA: False', reinicia el kernel y vuelve a ejecutar
import subprocess, sys, os
from pathlib import Path

# Verificar nvidia-smi
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version,utilization.gpu',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('[nvidia-smi]', r.stdout.strip())
else:
    print('nvidia-smi no encontrado')

# Verificar PyTorch
import torch
print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {props.name}')
    print(f'VRAM            : {props.total_memory / 1024**3:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')
    print()
    print('OK: GPU lista para entrenar')
else:
    print()
    print('PROBLEMA: PyTorch no detecta CUDA')
    print('Solucion: reinicia el kernel (Kernel -> Restart Kernel)')
    print('Si persiste, verifica con:')
    print('  pip show torch | findstr Version')
    print('  # Debe decir: torch 2.5.1+cu121  (no +cpu)')

In [ ]:
# ── CELDA 2: Instalar dependencias ───────────────────────────────
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics', 'seaborn', 'scikit-learn'
], check=False)
print('OK: dependencias listas')

In [ ]:
# ── CELDA 3: Configuracion de rutas ──────────────────────────────
# ─── CAMBIAR AQUI ─────────────────────────────────────────────────
COLAB_MODE  = False   # False = Local | True = Google Colab
FORCE_GPU   = True    # True = error si no hay GPU (evita entrenar en CPU por error)
# ──────────────────────────────────────────────────────────────────

if COLAB_MODE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE        = Path('/content/drive/MyDrive/ZapalloAI')
    DATA_DIR    = str(BASE / 'dataset_processed')
    PROJECT_DIR = str(BASE / 'runs' / 'classify')
    EXPORT_DIR  = BASE / 'exports'
else:
    ROOT = Path(os.path.abspath('')).resolve()
    for _ in range(6):
        if (ROOT / 'model').exists() and (ROOT / 'zapallo_app').exists():
            break
        ROOT = ROOT.parent
    DATA_DIR    = str(ROOT / 'model' / 'data' / 'processed')
    PROJECT_DIR = str(ROOT / 'model' / 'runs' / 'classify')
    EXPORT_DIR  = ROOT / 'model' / 'exports'
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ['healthy', 'downy_mildew', 'leaf_curl', 'mosaic_virus', 'red_beetle']

# Verificar GPU
import torch
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

if FORCE_GPU and DEVICE == 'cpu':
    raise RuntimeError(
        'GPU no detectada por PyTorch.\n'
        'Solucion: Kernel -> Restart Kernel y ejecuta de nuevo.\n'
        'O cambia FORCE_GPU = False para usar CPU (mucho mas lento).'
    )

print(f'Modo        : {"Google Colab" if COLAB_MODE else "Local"}')
print(f'Dispositivo : {"GPU: " + torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU"}')
if DEVICE == 0:
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM        : {vram:.1f} GB')
print(f'Dataset     : {DATA_DIR}')
print(f'Runs        : {PROJECT_DIR}')

# Verificar dataset
data_path = Path(DATA_DIR)
if data_path.exists():
    for s in ['train', 'val', 'test']:
        n = sum(1 for _ in (data_path / s).rglob('*.*')) if (data_path / s).exists() else 0
        print(f'  {s:<5}: {n:,} imgs')
else:
    print('ERROR: Dataset no encontrado ->', DATA_DIR)

In [ ]:
# ── CELDA 4: Distribucion del dataset ────────────────────────────
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({'font.family': 'DejaVu Sans'})

counts = {split: {} for split in ['train', 'val', 'test']}
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        p = Path(DATA_DIR) / split / cls
        counts[split][cls] = sum(1 for _ in p.glob('*.*')) if p.exists() else 0

print(f"{'Clase':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print('-' * 50)
for cls in CLASSES:
    t, v, ts = counts['train'][cls], counts['val'][cls], counts['test'][cls]
    print(f'{cls:<20} {t:>8} {v:>8} {ts:>8} {t+v+ts:>8}')

x    = range(len(CLASSES))
t_v  = [counts['train'][c] for c in CLASSES]
v_v  = [counts['val'][c]   for c in CLASSES]
ts_v = [counts['test'][c]  for c in CLASSES]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, t_v,  label='Train', color='#2D6A4F')
ax.bar(x, v_v,  bottom=t_v, label='Val', color='#52B788')
ax.bar(x, ts_v, bottom=[a+b for a,b in zip(t_v,v_v)], label='Test', color='#95D5B2')
ax.set_xticks(list(x)); ax.set_xticklabels(CLASSES, rotation=15, ha='right')
ax.set_ylabel('Imagenes'); ax.set_title('Dataset ZapalloAI'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── CELDA 5: ENTRENAMIENTO CON GPU ────────────────────────────────
from ultralytics import YOLO
import torch

# Verificar GPU antes de empezar
assert torch.cuda.is_available(), 'GPU no disponible. Reinicia el kernel.'

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Parametros optimizados para GTX 1650 (4 GB VRAM)
# amp=True reduce uso de VRAM a la mitad con precision mixta
EPOCHS  = 100
BATCH   = 16     # Seguro para 4 GB. Bajar a 8 si hay OOM
WORKERS = 4

print(f'GPU     : {GPU_NAME}')
print(f'VRAM    : {VRAM_GB:.1f} GB')
print(f'Epochs  : {EPOCHS}')
print(f'Batch   : {BATCH}')
print(f'AMP     : True (mixed precision activo)')
print(f'Datos   : {DATA_DIR}')
print()
print('Iniciando entrenamiento...')
print('Tiempo estimado: ~1.5 - 2 horas en GTX 1650')

# Modelo base pre-entrenado en ImageNet
model = YOLO('yolo11n-cls.pt')

results = model.train(
    data      = DATA_DIR,
    epochs    = EPOCHS,
    imgsz     = 224,
    batch     = BATCH,
    patience  = 15,           # Early stopping si no mejora en 15 epochs
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    cos_lr    = True,
    augment   = True,
    degrees   = 30,
    fliplr    = 0.5,
    flipud    = 0.3,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    erasing   = 0.4,
    mixup     = 0.1,
    project   = PROJECT_DIR,
    name      = 'zapallo_yolov11n_v1',
    exist_ok  = True,
    device    = 0,            # <-- GPU 0 (GTX 1650), no 'cpu'
    workers   = WORKERS,
    amp       = True,         # Mixed precision: menos VRAM, mas rapido
    cache     = False,        # No cachear en RAM
    verbose   = True,
)

BEST_MODEL = Path(PROJECT_DIR) / 'zapallo_yolov11n_v1' / 'weights' / 'best.pt'
print(f'\nEntrenamiento completado!')
print(f'Mejor modelo: {BEST_MODEL}')
print(f'Top-1 val accuracy: {results.results_dict.get("metrics/accuracy_top1", "N/A")}')

In [ ]:
# ── CELDA 6: Evaluacion en test set ──────────────────────────────
from ultralytics import YOLO
import torch

BEST_MODEL = Path(PROJECT_DIR) / 'zapallo_yolov11n_v1' / 'weights' / 'best.pt'
assert BEST_MODEL.exists(), f'Ejecuta primero la celda de entrenamiento. No encontrado: {BEST_MODEL}'

model   = YOLO(str(BEST_MODEL))
metrics = model.val(data=DATA_DIR, split='test', imgsz=224, batch=16, device=0)

print('Metricas en Test Set:')
print(f'  Top-1 Accuracy : {metrics.top1*100:.2f}%')
print(f'  Top-5 Accuracy : {metrics.top5*100:.2f}%')

In [ ]:
# ── CELDA 7: Matriz de confusion ─────────────────────────────────
import numpy as np, seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO

BEST_MODEL = Path(PROJECT_DIR) / 'zapallo_yolov11n_v1' / 'weights' / 'best.pt'
assert BEST_MODEL.exists(), 'Ejecuta primero el entrenamiento'

model = YOLO(str(BEST_MODEL))
all_preds, all_labels = [], []

for cls_idx, cls_name in enumerate(CLASSES):
    imgs = list((Path(DATA_DIR) / 'test' / cls_name).glob('*.jpg'))
    imgs += list((Path(DATA_DIR) / 'test' / cls_name).glob('*.png'))
    for img_path in imgs:
        res = model(str(img_path), verbose=False)[0]
        all_preds.append(res.probs.top1)
        all_labels.append(cls_idx)

print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))

cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Matriz de Confusion (conteos)')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predicho')
cm_n = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title('Matriz de Confusion (normalizada)')
axes[1].set_ylabel('Real'); axes[1].set_xlabel('Predicho')
plt.tight_layout()
fig_path = EXPORT_DIR / 'confusion_matrix.png'
plt.savefig(str(fig_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {fig_path}')

In [ ]:
# ── CELDA 8: Exportar a TFLite ───────────────────────────────────
from ultralytics import YOLO
import shutil, glob

BEST_MODEL = Path(PROJECT_DIR) / 'zapallo_yolov11n_v1' / 'weights' / 'best.pt'
assert BEST_MODEL.exists(), 'Ejecuta primero el entrenamiento'

model = YOLO(str(BEST_MODEL))

print('Exportando float32...')
model.export(format='tflite', imgsz=224)

print('Exportando int8 (para movil)...')
model.export(format='tflite', int8=True, imgsz=224, data=DATA_DIR)

# Copiar a exports/
for f in glob.glob(str(BEST_MODEL.parent / '**' / '*.tflite'), recursive=True):
    dst = EXPORT_DIR / Path(f).name
    shutil.copy2(f, dst)
    print(f'  {Path(f).name} -> {dst} ({Path(f).stat().st_size/1024**2:.2f} MB)')

print(f'\nExports en: {EXPORT_DIR}')

In [ ]:
# ── CELDA 9: Generar labels.txt ──────────────────────────────────
labels = ['healthy', 'downy_mildew', 'leaf_curl', 'mosaic_virus', 'red_beetle']
lbl_path = EXPORT_DIR / 'labels.txt'
lbl_path.write_text('\n'.join(labels), encoding='utf-8')

print('labels.txt generado:')
for i, l in enumerate(labels):
    print(f'  {i}: {l}')
print()
print('Copiar a la app Flutter:')
print('  model/exports/best_int8.tflite -> zapallo_app/assets/models/')
print('  model/exports/labels.txt       -> zapallo_app/assets/models/')

## Resumen

| Item | Valor |
|---|---|
| GPU | GTX 1650 4 GB |
| Modelo | YOLOv11n-cls |
| Epochs | 100 (early stop patience=15) |
| Batch | 16 (AMP activado) |
| Tiempo estimado | ~1.5 - 2 horas |
| Objetivo accuracy | >= 92% |

### Si hay OOM (Out of Memory)
En la celda 5, cambiar:
```python
BATCH = 8
```

### Siguiente paso
Integrar `best_int8.tflite` en la app Flutter (Sprint 4).